In [1]:
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np
from typing import List, Dict, Tuple

print("Day 15 - Cross-Encoder Re-ranking")

# Bi-encoder -- fast, used for retrieval
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")

# Cross-encoder -- slower, used for re-ranking
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Bi-encoder loaded")
print("Cross-encoder loader")

Day 15 - Cross-Encoder Re-ranking
Bi-encoder loaded
Cross-encoder loader


In [2]:
print("=== Bi-Encoder vs Cross-Encoder ===\n")

query = "how does retrieval work in RAG?"

chunks = [
    "RAGAs evaluates faithfulness by checking if answers are grounded in context",
    "Hybrid search combines BM25 and vector search merged with RRF for retrieval",
    "Cross-encoder re-rankers process query and document together for precision",
    "FastAPI handles authentication and async request routing for the backend",
    "BM25 ranks documents based on keyword frequency and inverse document frequency"
]

# ----- Bi-encoder approach ----
# Encodes query and documents seperately
# Then computes cosine similarity between vectors
print("=== Bi-Encoder (how your retriever works) ===")
query_embedding = bi_encoder.encode(query)
chunk_embeddings = bi_encoder.encode(chunks)

# Cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
bi_scores = cosine_similarity([query_embedding], chunk_embeddings)[0]

print(f"Query encoded separately: shape {query_embedding.shape}")
print(f"Chunks encoded separately: shape {chunk_embeddings.shape}")
print("\nBi-encoder scores:")
bi_ranked = sorted(enumerate(bi_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(bi_ranked):
    print(f"  Rank {rank+1}: [{score:.4f}] {chunks[idx][:60]}...")

# ---- Cross-encoder approach ---
# Encodes query AND document TOGETHER
# Read full interaction between them
print("\n=== Cross-Encoder (how re-ranking works) ===")
query_chunk_pairs = [[query, chunk] for chunk in chunks]
cross_scores = cross_encoder.predict(query_chunk_pairs)

print(f"Query+chunk pairs evaluated: {len(query_chunk_pairs)}")
print("\nCross-encoder scores:")
cross_ranked = sorted(enumerate(cross_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(cross_ranked):
    print(f"  Rank {rank+1}: [{score:.4f}] {chunks[idx][:60]}...")
    
print("\n=== Key Difference ===")
print("Bi-encoder:   query → vector, doc → vector, compare separately")
print("Cross-encoder: [query + doc] → relevance score, evaluated together")
print("Cross-encoder is slower but far more accurate")




=== Bi-Encoder vs Cross-Encoder ===

=== Bi-Encoder (how your retriever works) ===
Query encoded separately: shape (384,)
Chunks encoded separately: shape (5, 384)

Bi-encoder scores:
  Rank 1: [0.3361] RAGAs evaluates faithfulness by checking if answers are grou...
  Rank 2: [0.1971] Hybrid search combines BM25 and vector search merged with RR...
  Rank 3: [0.1744] Cross-encoder re-rankers process query and document together...
  Rank 4: [0.0990] BM25 ranks documents based on keyword frequency and inverse ...
  Rank 5: [0.0827] FastAPI handles authentication and async request routing for...

=== Cross-Encoder (how re-ranking works) ===
Query+chunk pairs evaluated: 5

Cross-encoder scores:
  Rank 1: [-3.4498] RAGAs evaluates faithfulness by checking if answers are grou...
  Rank 2: [-9.8614] Hybrid search combines BM25 and vector search merged with RR...
  Rank 3: [-11.3703] Cross-encoder re-rankers process query and document together...
  Rank 4: [-11.4657] FastAPI handles authenticat

In [5]:
print("=== Clear Re-ranking Demonstration ===\n")

query = "what is the capital of france?"

# Mix of relevant and irrelevant chunks 
test_chunks = [
    "Python is a programming language used for data science and ML",
    "Paris is the capital city of France and its largest city",
    "Machine learning model require large amounts of training data",
    "France is a country in western Europe. Its capital is Paris",
    "Neural networks are inspired by the human brain strucure"
]

# Bi-encoder
query_emb = bi_encoder.encode(query)
chunk_embs = bi_encoder.encode(test_chunks)
bi_scores = cosine_similarity([query_emb], chunk_embs)[0]

print("Bi-encoder ranking:")
bi_ranked = sorted(enumerate(bi_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(bi_ranked):
    print(f"  Rank {rank+1}: [{score:.4f}] {test_chunks[idx][:60]}...")

# Cross-encoder
pairs = [[query, chunk] for chunk in test_chunks]
cross_encoder = cross_encoder.predict(pairs)

print("\nCross-encoder ranking:")
cross_ranked = sorted(enumerate(cross_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(cross_ranked):
    print(f"  Rank {rank+1}: [{score:.4f}] {test_chunks[idx][:60]}...")

print("\nExpected: Paris/France chunks should rank 1 and 2")

=== Clear Re-ranking Demonstration ===

Bi-encoder ranking:
  Rank 1: [0.8017] France is a country in western Europe. Its capital is Paris...
  Rank 2: [0.7517] Paris is the capital city of France and its largest city...
  Rank 3: [0.1028] Python is a programming language used for data science and M...
  Rank 4: [0.0458] Machine learning model require large amounts of training dat...
  Rank 5: [0.0081] Neural networks are inspired by the human brain strucure...

Cross-encoder ranking:
  Rank 1: [-3.4498] Python is a programming language used for data science and M...
  Rank 2: [-9.8614] Paris is the capital city of France and its largest city...
  Rank 3: [-11.3703] Machine learning model require large amounts of training dat...
  Rank 4: [-11.4657] France is a country in western Europe. Its capital is Paris...
  Rank 5: [-11.4766] Neural networks are inspired by the human brain strucure...

Expected: Paris/France chunks should rank 1 and 2


In [7]:
from sentence_transformers import CrossEncoder

# Reload cross encoder
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Test immediately
test_pairs = [
    ["what is the capital of France?", "Paris is the capital city of France"],
    ["what is the capital of France?", "Python is a programming language"]
]

scores = cross_encoder.predict(test_pairs)
print(f"Paris chunk score: {scores[0]:.4f}")
print(f"Python chunk score: {scores[1]:.4f}")
print(f"Paris should score much higher than Python")

Paris chunk score: 7.7254
Python chunk score: -11.0701
Paris should score much higher than Python


In [ ]:
print("=== Correct Re-ranking Demonstration ===\n")

query = "how does retrieval work in RAG? "

chunks = [
    "RAGAs evaluates faithfulness by checking if answers are grounded in context",
    "Hybrid search combines BM25 and vector search merged with RRF for retrieval",
    "Cross-encoder re-rankers process query and document together for precision",
    "FastAPI handles authentication and async request routing for the backend",
    "BM25 ranks documents based on keyword frequency and inverse document frequency"
]

# Bi-encoder
query_emb = bi_encoder.encode(query)
chunk_embs = bi_encoder.encode(chunks)
bi_scores = cosine_similarity([query_emb], chunk_embs)[0]

print("Bi-encoder ranking:")
bi_ranked = sorted(enumerate(bi_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(bi_ranked):
    print(f"  Rank {rank+1}: [{score:.4f}] {chunks[idx][:60]}...")

# Cross-encoder - use different variable name for scores
ce_pairs = [[query, chunk] for chunk in chunks]
ce_scores = cross_encoder.predict(ce_pairs)

print("\nCross-encoder ranking:")
ce_ranked = sorted(enumerate(ce_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(ce_ranked):
    print(f"  Rank {rank+1}: [{score:.4f}] {chunks[idx][:60]}...")

#  Show ranking changes
print("\n=== Ranking Changes ===")
bi_order = [idx for idx, _ in bi_ranked]
ce_order = [idx for idx, _ in ce_ranked]

for ce_rank, idx in enumerate(ce_order):
    bi_rank = bi_order.index(idx)
    change = bi_rank - ce_rank
    arrow = "^"* change if change > 0 else "↓" * abs(change) if change < 0 else "->"
    print(f"   {arrow} '{chunks[idx][:45]}...' (bi:{bi_rank+1} -> ce:{ce_rank+1})")

=== Correct Re-ranking Demonstration ===

Bi-encoder ranking:
  Rank 1: [0.3361] RAGAs evaluates faithfulness by checking if answers are grou...
  Rank 2: [0.1971] Hybrid search combines BM25 and vector search merged with RR...
  Rank 3: [0.1744] Cross-encoder re-rankers process query and document together...
  Rank 4: [0.0990] BM25 ranks documents based on keyword frequency and inverse ...
  Rank 5: [0.0827] FastAPI handles authentication and async request routing for...

Cross-encoder ranking:
  Rank 1: [-3.4498] RAGAs evaluates faithfulness by checking if answers are grou...
  Rank 2: [-9.8614] Hybrid search combines BM25 and vector search merged with RR...
  Rank 3: [-11.3703] Cross-encoder re-rankers process query and document together...
  Rank 4: [-11.4657] FastAPI handles authentication and async request routing for...
  Rank 5: [-11.4766] BM25 ranks documents based on keyword frequency and inverse ...

=== Ranking Changes ===
   -> 'RAGAs evaluates faithfulness by checking if 

In [9]:
print("=== Re-ranking With Clear Relevance Differences ===\n")

query = "how does BM25 keyword search work?"

# Mix of highly relevant, partially relevant, and irrelevant
chunks = [
    "BM25 ranks documents using term frequency and inverse document frequency. It gives higher scores to documents where query terms appear frequently but are rare across the corpus.",
    "Search engines use various algorithms to retrieve documents from large collections of text data stored in databases.",
    "BM25 stands for Best Match 25. The algorithm penalizes very long documents to normalize scores across different document lengths.",
    "Neural networks process information through layers of interconnected nodes called neurons that transform input signals.",
    "Keyword matching algorithms like BM25 score zero for documents that do not contain any query terms making it precise for exact lookups.",
    "FastAPI is a modern Python web framework for building REST APIs with automatic documentation generation."
]

# Bi-encoder scores
query_emb = bi_encoder.encode(query)
chunk_embs = bi_encoder.encode(chunks)
bi_scores = cosine_similarity([query_emb], chunk_embs)[0]

print("Bi-encoder ranking:")
bi_ranked = sorted(enumerate(bi_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(bi_ranked):
    print(f"  Rank {rank+1}: [{score:.4f}] {chunks[idx][:65]}...")

# Cross-encoder scores
ce_pairs = [[query, chunk] for chunk in chunks]
ce_scores = cross_encoder.predict(ce_pairs)

print("\nCross-encoder ranking:")
ce_ranked = sorted(enumerate(ce_scores), key=lambda x: x[1], reverse=True)
for rank, (idx, score) in enumerate(ce_ranked):
    print(f"  Rank {rank+1}: [{score:.4f}] {chunks[idx][:65]}...")

# Ranking changes
print("\n=== Ranking Changes ===")
bi_order = [idx for idx, _ in bi_ranked]
ce_order = [idx for idx, _ in ce_ranked]
for ce_rank, idx in enumerate(ce_order):
    bi_rank = bi_order.index(idx)
    change = bi_rank - ce_rank
    arrow = "↑"*change if change > 0 else "↓"*abs(change) if change < 0 else "→"
    print(f"  {arrow} Chunk {idx}: '{chunks[idx][:50]}...'")

=== Re-ranking With Clear Relevance Differences ===

Bi-encoder ranking:
  Rank 1: [0.6360] BM25 ranks documents using term frequency and inverse document fr...
  Rank 2: [0.6359] Keyword matching algorithms like BM25 score zero for documents th...
  Rank 3: [0.6202] BM25 stands for Best Match 25. The algorithm penalizes very long ...
  Rank 4: [0.4802] Search engines use various algorithms to retrieve documents from ...
  Rank 5: [0.1576] FastAPI is a modern Python web framework for building REST APIs w...
  Rank 6: [0.0793] Neural networks process information through layers of interconnec...

Cross-encoder ranking:
  Rank 1: [4.3120] Keyword matching algorithms like BM25 score zero for documents th...
  Rank 2: [3.8277] BM25 ranks documents using term frequency and inverse document fr...
  Rank 3: [2.2961] BM25 stands for Best Match 25. The algorithm penalizes very long ...
  Rank 4: [-9.3817] Search engines use various algorithms to retrieve documents from ...
  Rank 5: [-11.3958] N

In [23]:
import os
import time
import uuid
import chromadb
from groq import Groq
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field
from datetime import datetime

load_dotenv()

class EnterpriseRAGPipelineV2:
    """ 
    Enterprise RAG Pipeline with cross-encoder re-ranking.
    Adds re-ranking layer after hybrid retrieval.
    """

    def __init__(
            self,
            org_id: str,
            groq_client,
            embedder: SentenceTransformer,
            reranker: CrossEncoder,
            persist_path: str="./rag_v2_db",
            model: str = "llama-3.1-8b-instant",
            temperature: float = 0.1,
            n_retrieve: int=6,
            n_rerank: int =3
    ):
        self.org_id = org_id
        self.client = groq_client
        self.embedder = embedder
        self.reranker = reranker
        self.model = model
        self.temperature = temperature
        self.n_retrieve = n_retrieve
        self.n_rerank = n_rerank

        # ChromaDB
        self.chroma_client = chromadb.PersistentClient(path=persist_path)
        self.collection = self.chroma_client.get_or_create_collection(
            name=f"org_{org_id}"
        )

        # BM25
        self.bm25 =None
        self.bm25_corpus = []
        self.bm25_ids = []
        self.bm25_metadatas = []

        # Conversation memory
        self.conversation_history = []

        self.system_prompt = """You are an Enterprise RAG assistant.
Answer questions based on the provided document chunks.
Always cite which chunk you used (e.g. 'According to chunk 1...').
Use your reasoning to connect related information in the chunks.
If there is truly no relevant information say:
'I cannot find this information in the provided documents.'
Be concise and professional."""

        print(f"[{org_id}] Pipeline V2 initialized with re-ranking")

    def ingest(self, texts, metadatas, ids=None):
        if ids is None:
            ids = [str(uuid.uuid4())[:8] for _ in texts]
        embeddings = self.embedder.encode(texts).tolist()
        self.collection.add(
            ids=ids, embeddings=embeddings,
            documents = texts, metadatas=metadatas
        )
        self.bm25_corpus.extend(texts)
        self.bm25_ids.extend(ids)
        self.bm25_metadatas.extend(metadatas)
        tokenized = [doc.lower().split() for doc in self.bm25_corpus]
        self.bm25 = BM25Okapi(tokenized)
        print(f"[{self.org_id}] Ingested {len(texts)} docs")

    def _vector_search(self, query, n):
        qe = self.embedder.encode(query).tolist()
        results = self.collection.query(
            query_embeddings=[qe],
            n_results = min(n, self.collection.count())
        )
        return[(results['ids'][0][i], 1 -results['distances'][0][i])
               for i in range(len(results['ids'][0]))]
    
    def _bm25_search(self, query, n):
        if self.bm25 is None:
            return[]
        scores =self.bm25.get_scores(query.lower().split())
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse = True)[:n]
        return [(self.bm25_ids[idx], score) for idx, score in ranked]
    
    def _rrf(self, result_sets, k=60):
        rrf_scores = {}
        for result_set in result_sets:
            for rank, (doc_id, _) in enumerate(result_set):
                rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) +1.0 /(k+rank+1)
        return sorted(rrf_scores.items(), key=lambda x:x[1], reverse=True)
    
    def _retrieve(self, query):
        """Hybrid retrieval - retrive more candidates for re-ranking"""
        vecotr_results = self._vector_search(query, self.n_retrieve)
        bm25_results = self._bm25_search(query, self.n_retrieve)
        merged = self._rrf([vecotr_results, bm25_results])

        chunks = []
        for doc_id, score in merged:
            if doc_id in self.bm25_ids:
                idx = self.bm25_ids.index(doc_id)
                chunks.append({
                    "id": doc_id,
                    "text": self.bm25_corpus[idx],
                    "score": score,
                    "metadata": self.bm25_metadatas[idx]
                })
        return chunks
    
    def _rerank(self, query: str, chunks: List[Dict]) -> List[Dict]:
        """Re-rank chunks using cross-encoder"""
        if not chunks:
            return chunks
        
        # Create query-chunk pairs
        pairs = [[query, chunk["text"]] for chunk in chunks]

        # Score with cross-encoder
        ce_scores = self.reranker.predict(pairs)

        # Add score for chunks and sort
        for i, chunk in enumerate(chunks):
            chunk["rerank_score"] = float(ce_scores[i])

        reranked = sorted(chunks, key=lambda x: x["rerank_score"], reverse=True)
        return reranked[:self.n_rerank]
    
    def _build_prompt(self, query, chunks): 
        context_parts = []
        for i, chunk in enumerate(chunks):
            rerank_score = chunk.get("rerank_score", chunk["score"])
            context_parts.append(
                f"Chunk {i+1} [{chunk['metadata'].get('source', 'unknown')} | "
                f"relevance: {rerank_score:.2f}]:\n{chunk['text']}"
            )
        context = "\n\n".join(context_parts)
        return f"Document chunks:\n{context}\n\nQuestion: {query}"
    
    def _generate(self, user_message):
        messages = [{"role": "system", "content": self.system_prompt}]
        messages.extend(self.conversation_history)
        messages.append({"role": "user", "content": user_message})
        response = self.client.chat.completions.create(
            model = self.model,
            messages = messages,
            temperature =self.temperature,
            max_tokens = 500
        )
        tokens = 0
        if response.usage:
            tokens =response.usage.prompt_tokens + response.usage.completion_tokens
        return response.choices[0].message.content, tokens
    
    def query(self, user_query: str, use_memory: bool =True) -> Dict:
        """Full pipeline: hybrid retrieve -> rerank -> generate"""
        start = time.time()

        # Step 1 - Hybrid retrival (more candidates)
        candidates = self._retrieve(user_query)

        # Step 2 - Re-rank (filter to best)
        reranked_chunks = self._rerank(user_query, candidates)

        # Step 3 - Generate
        user_message = self._build_prompt(user_query, reranked_chunks)
        answer, tokens = self._generate(user_message)

        # Step 4 - Memoery
        if use_memory:
            self.conversation_history.append(
                {"role": "user", "content": user_query}
            )
            self.conversation_history = self.conversation_history[-6:]

        latency = (time.time() - start) * 1000

        return {
            "query": user_query,
            "answer": answer,
            "chunks_retrieved": len(candidates),
            "chunks_after_rerank": len(reranked_chunks),
            "top_chunk_rerank_score": reranked_chunks[0]["rerank_score"] if reranked_chunks else 0,
            "latency_ms": latency,
            "tokens": tokens
        }
    
print("EnterpriseRAGpipelineV2 defined")

EnterpriseRAGpipelineV2 defined


In [24]:
groq_client = Groq(api_key=os.getenv("ROQ_API_KEY"))

# Initialize V2 pipeline
pipeline_v2 = EnterpriseRAGPipelineV2(
    org_id = "acme_corp",
    groq_client = groq_client,
    embedder =bi_encoder,
    reranker=cross_encoder,
    persist_path="./rag_v2_db"
)

# Ingest Same knowledge base
pipeline_v2.ingest(
    texts =[
        "Hybrid search combines BM25 keyword search with vector semantic search. Results are merged using Reciprocal Rank Fusion which uses rank position not raw scores.",
        "RAGAs evaluates RAG pipelines using four metrics: faithfulness, answer relevancy, context recall and context precision. Faithfulness measures if answers are grounded in context.",
        "Multi-tenancy is implemented through ChromaDB collection isolation. Each organization gets a separate collection. Users cannot access other organizations data.",
        "Re-ranking uses a cross-encoder model to re-score retrieved chunks. Unlike bi-encoders, cross-encoders process query and document together for higher precision.",
        "LoRA fine-tuning reduces trainable parameters by 90 percent using low-rank matrix decomposition. It adds small adapter matrices to existing weights.",
        "FastAPI provides automatic OpenAPI documentation, async request handling, and Pydantic data validation for ML backends.",
        "Docker containerization ensures consistent environments. docker-compose manages FastAPI, ChromaDB, Redis and Celery services together.",
        "The pipeline is deployed on AWS EC2 with Nginx as reverse proxy. GitHub Actions handles CI/CD on push to main branch.",
        "Langfuse provides LLM observability — traces every call showing retrieved chunks, tokens used, latency and model parameters.",
        "Celery with Redis handles async document ingestion. Large PDFs process in background — user gets a job_id to poll status."
    ],
    metadatas = [
        {"source": "architecture.pdf", "topic": "retrieval"},
        {"source": "evaluation.pdf", "topic": "evaluation"},
        {"source": "architecture.pdf", "topic": "multitenancy"},
        {"source": "retrieval.pdf", "topic": "reranking"},
        {"source": "training.pdf", "topic": "finetuning"},
        {"source": "backend.pdf", "topic": "api"},
        {"source": "deployment.pdf", "topic": "docker"},
        {"source": "deployment.pdf", "topic": "cloud"},
        {"source": "monitoring.pdf", "topic": "observability"},
        {"source": "backend.pdf", "topic": "async"}
    ]
)

# Test queries
print("\n=== Testing Pipeline V2 ===\n")

queries = {
    "How does hybrid search work?",
    "What metrics does RAGAs use for evaluation?",
    "How is re-ranking different from retrieval?",
    "what is the pricing for the enterprise plan?"
}

for query in queries:
    result= pipeline_v2.query(query, use_memory=False)
    print(f"Q: {query}")
    print(f"A: {result['answer'][:120]}...")
    print(f"   Retrieved: {result['chunks_retrieved']} → Reranked: {result['chunks_after_rerank']}")
    print(f"   Top rerank score: {result['top_chunk_rerank_score']:.4f}")
    print(f"   Latency: {result['latency_ms']:.0f}ms | Tokens: {result['tokens']}")
    print()


[acme_corp] Pipeline V2 initialized with re-ranking
[acme_corp] Ingested 10 docs

=== Testing Pipeline V2 ===

Q: How is re-ranking different from retrieval?
A: According to Chunk 1, re-ranking uses a cross-encoder model to re-score retrieved chunks. This implies that retrieval is...
   Retrieved: 6 → Reranked: 3
   Top rerank score: 4.5511
   Latency: 822ms | Tokens: 355

Q: What metrics does RAGAs use for evaluation?
A: According to chunk 1, RAGAs evaluates RAG pipelines using four metrics: faithfulness, answer relevancy, context recall, ...
   Retrieved: 6 → Reranked: 3
   Top rerank score: 8.8842
   Latency: 518ms | Tokens: 291

Q: what is the pricing for the enterprise plan?
A: I cannot find this information in the provided documents....
   Retrieved: 6 → Reranked: 3
   Top rerank score: -11.3384
   Latency: 495ms | Tokens: 264

Q: How does hybrid search work?
A: According to chunk 1, hybrid search combines two types of search: BM25 keyword search and vector semantic search. The r

In [25]:
# Debug cell
candidates = pipeline_v2._retrieve("How is re-ranking different from retrieval?")
reranked = pipeline_v2._rerank("How is re-ranking different from retrieval?", candidates)

print("Top 3 chunks after reranking:")
for i, chunk in enumerate(reranked):
    print(f"\nChunk {i+1} [score: {chunk['rerank_score']:.4f}]:")
    print(f"  {chunk['text']}")

Top 3 chunks after reranking:

Chunk 1 [score: 4.5511]:
  Re-ranking uses a cross-encoder model to re-score retrieved chunks. Unlike bi-encoders, cross-encoders process query and document together for higher precision.

Chunk 2 [score: -10.0491]:
  Hybrid search combines BM25 keyword search with vector semantic search. Results are merged using Reciprocal Rank Fusion which uses rank position not raw scores.

Chunk 3 [score: -10.9292]:
  RAGAs evaluates RAG pipelines using four metrics: faithfulness, answer relevancy, context recall and context precision. Faithfulness measures if answers are grounded in context.


In [26]:
result = pipeline_v2.query(
    "How is re-ranking different from retrieval?",
    use_memory=False
)
print(result['answer'])

According to Chunk 1, re-ranking uses a cross-encoder model to re-score retrieved chunks. This implies that retrieval is the initial process of retrieving chunks, and re-ranking is a subsequent step that refines the ranking of these retrieved chunks.

In other words, retrieval is the process of finding relevant chunks, while re-ranking is the process of re-ordering these chunks based on their relevance to the query.
